In [1]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
len(words)

32033

In [3]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [4]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:3]:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
... ---> a
..a ---> v
.av ---> a
ava ---> .


In [5]:
X.shape, X.dtype, Y.shape, Y.dtype
# X: 16个样本，每个字符3个上下文

(torch.Size([16, 3]), torch.int64, torch.Size([16]), torch.int64)

In [6]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1]])

In [7]:
C = torch.randn((27,2))  # 2维，每个字母用2个参数表示

In [8]:
C

tensor([[ 1.2709,  0.1040],
        [ 0.2415,  0.1809],
        [-0.3435,  1.0362],
        [ 1.3137, -0.5120],
        [ 0.1503, -0.2539],
        [-0.6732, -0.0649],
        [ 0.1809, -0.2733],
        [ 0.0493,  0.5962],
        [-0.8485, -1.6294],
        [ 0.4327, -0.8266],
        [-0.0273,  1.6573],
        [ 0.9972,  0.2268],
        [-0.2858,  0.1246],
        [-1.2497, -1.4502],
        [-0.2341,  0.8451],
        [-2.1005,  0.7643],
        [ 2.4846,  0.0560],
        [-0.0358, -0.4579],
        [-2.2857,  1.0025],
        [-2.5640, -2.0975],
        [ 0.6732,  1.6387],
        [-0.2790,  0.8998],
        [-0.2752, -0.4325],
        [ 1.7707, -0.0026],
        [ 0.6905, -0.4799],
        [-0.1693,  2.3364],
        [ 0.3754, -0.0852]])

In [9]:
idx = torch.tensor([0, 2]) 
C[idx]

tensor([[ 1.2709,  0.1040],
        [-0.3435,  1.0362]])

In [10]:
C[X].shape # 对X每个元素查表，再堆叠

torch.Size([16, 3, 2])

In [11]:
C[X] # 等价于查表

tensor([[[ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [ 1.2709,  0.1040]],

        [[ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [-0.6732, -0.0649]],

        [[ 1.2709,  0.1040],
         [-0.6732, -0.0649],
         [-1.2497, -1.4502]],

        [[-0.6732, -0.0649],
         [-1.2497, -1.4502],
         [-1.2497, -1.4502]],

        [[-1.2497, -1.4502],
         [-1.2497, -1.4502],
         [ 0.2415,  0.1809]],

        [[ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [ 1.2709,  0.1040]],

        [[ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [-2.1005,  0.7643]],

        [[ 1.2709,  0.1040],
         [-2.1005,  0.7643],
         [-0.2858,  0.1246]],

        [[-2.1005,  0.7643],
         [-0.2858,  0.1246],
         [ 0.4327, -0.8266]],

        [[-0.2858,  0.1246],
         [ 0.4327, -0.8266],
         [-0.2752, -0.4325]],

        [[ 0.4327, -0.8266],
         [-0.2752, -0.4325],
         [ 0.4327, -0.8266]],

        [[-0.2752, -0

In [12]:
emb = C[X]
emb.shape

torch.Size([16, 3, 2])

In [13]:
# emb[sample, char_position, embedding_dim]                                          
#        ↑         ↑                ↑                                                  
#       dim0      dim1             dim2    
# emb[:, t, :] -> 所有样本的第 t 个字符
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], dim=1)

tensor([[ 1.2709,  0.1040,  1.2709,  0.1040,  1.2709,  0.1040],
        [ 1.2709,  0.1040,  1.2709,  0.1040, -0.6732, -0.0649],
        [ 1.2709,  0.1040, -0.6732, -0.0649, -1.2497, -1.4502],
        [-0.6732, -0.0649, -1.2497, -1.4502, -1.2497, -1.4502],
        [-1.2497, -1.4502, -1.2497, -1.4502,  0.2415,  0.1809],
        [ 1.2709,  0.1040,  1.2709,  0.1040,  1.2709,  0.1040],
        [ 1.2709,  0.1040,  1.2709,  0.1040, -2.1005,  0.7643],
        [ 1.2709,  0.1040, -2.1005,  0.7643, -0.2858,  0.1246],
        [-2.1005,  0.7643, -0.2858,  0.1246,  0.4327, -0.8266],
        [-0.2858,  0.1246,  0.4327, -0.8266, -0.2752, -0.4325],
        [ 0.4327, -0.8266, -0.2752, -0.4325,  0.4327, -0.8266],
        [-0.2752, -0.4325,  0.4327, -0.8266,  0.2415,  0.1809],
        [ 1.2709,  0.1040,  1.2709,  0.1040,  1.2709,  0.1040],
        [ 1.2709,  0.1040,  1.2709,  0.1040,  0.2415,  0.1809],
        [ 1.2709,  0.1040,  0.2415,  0.1809, -0.2752, -0.4325],
        [ 0.2415,  0.1809, -0.2752, -0.4

In [14]:
torch.unbind(emb, 1)

(tensor([[ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [-0.6732, -0.0649],
         [-1.2497, -1.4502],
         [ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [-2.1005,  0.7643],
         [-0.2858,  0.1246],
         [ 0.4327, -0.8266],
         [-0.2752, -0.4325],
         [ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [ 0.2415,  0.1809]]),
 tensor([[ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [-0.6732, -0.0649],
         [-1.2497, -1.4502],
         [-1.2497, -1.4502],
         [ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [-2.1005,  0.7643],
         [-0.2858,  0.1246],
         [ 0.4327, -0.8266],
         [-0.2752, -0.4325],
         [ 0.4327, -0.8266],
         [ 1.2709,  0.1040],
         [ 1.2709,  0.1040],
         [ 0.2415,  0.1809],
         [-0.2752, -0.4325]]),
 tensor([[ 1.2709,  0.1040],
         [-0.6732, -0.0649],
         [

In [15]:
a = torch.arange(12)
a.shape

torch.Size([12])

In [16]:
a.view(3, 2, 2)

tensor([[[ 0,  1],
         [ 2,  3]],

        [[ 4,  5],
         [ 6,  7]],

        [[ 8,  9],
         [10, 11]]])

In [17]:
a.storage()

C:\Users\Hi\AppData\Local\Temp\ipykernel_29196\214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 12]

In [18]:
# torch底层存储的是一维，我们可以高效地用view来调整tensor shape
emb.shape

torch.Size([16, 3, 2])

In [19]:
emb.view(16, 6).shape

torch.Size([16, 6])

In [20]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [21]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # -1: 让torch自己推断
# emb 输入每个都是6维
# @ W1 -> 输出 100 维
h

tensor([[-0.9856, -0.7565, -0.5614,  ...,  0.9865,  0.0247, -0.9996],
        [ 0.9431,  0.5790,  0.7743,  ...,  0.9883,  0.4874, -0.9874],
        [ 0.9939,  0.4452,  0.3751,  ...,  0.9996,  0.8309,  0.7697],
        ...,
        [-0.4360, -0.1382,  0.1470,  ...,  0.9850,  0.0899, -0.9979],
        [ 0.1884, -0.2370, -0.1839,  ...,  0.9984,  0.3963, -0.8720],
        [-0.7014, -0.7623, -0.6341,  ...,  0.3370, -0.8933, -0.7248]])

In [22]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [23]:
logits = h @ W2 + b2

In [24]:
logits.shape

torch.Size([16, 27])

In [25]:
counts = logits.exp() # exp 的导数就是它本身，方便计算
probs = counts / counts.sum(1, keepdims=True)

In [26]:
probs.shape # 16个样本，每个输入对应27个预测分布

torch.Size([16, 27])

In [27]:
probs[0].sum()

tensor(1.)

In [28]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0])

In [29]:
torch.arange(16)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [30]:
probs[torch.arange(16), Y]
# 每个索引管一个维度
# (arange(16)[0], Y[0]) = (0, Y[0])   →  probs[0, Y[0]]
# (arange(16)[1], Y[1]) = (1, Y[1])   →  probs[1, Y[1]]
# ...
# (arange(16)[15], Y[15]) = (15, Y[15]) →  probs[15, Y[15]]
# 拿到正确答案Y的概率

tensor([9.6534e-01, 2.7887e-07, 2.3404e-05, 2.3165e-13, 1.0493e-05, 7.3368e-11,
        3.5845e-12, 1.0014e-06, 1.1026e-06, 8.1571e-03, 4.9144e-03, 6.0148e-05,
        1.1025e-07, 1.8522e-02, 1.5833e-07, 4.6097e-07])

In [31]:
probs[torch.arange(16), Y].log() # log相乘便利性，方便计算mean

tensor([ -0.0353, -15.0925, -10.6626, -29.0935, -11.4648, -23.3355, -26.3544,
        -13.8141, -13.7178,  -4.8089,  -5.3156,  -9.7187, -16.0205,  -3.9888,
        -15.6586, -14.5899])

In [32]:
loss = -probs[torch.arange(16), Y].log().mean() # 平均惊讶度
loss

tensor(13.3545)

In [33]:
# make it clear
X.shape, Y.shape

(torch.Size([16, 3]), torch.Size([16]))

In [34]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator = g)
W1 = torch.randn((6, 100), generator = g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [35]:
sum(p.nelement() for p in parameters)

3481

In [36]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)
loss = -probs[torch.arange(16), Y].log().mean() 
loss

tensor(20.2417)

In [37]:
F.cross_entropy(logits, Y)

tensor(20.2417)

In [42]:
for p in parameters:
    p.requires_grad = True

In [43]:
for _ in range(20):
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    print(loss.item())
    for p in parameters:
        p.grad = None
    loss.backward()
    for p in parameters:
        p.data += -0.1 * p.grad

20.24171257019043
14.233867645263672
10.578411102294922
7.81959867477417
6.053427696228027
4.864985466003418
3.8876330852508545
3.076685667037964
2.403904914855957
1.8741711378097534
1.4853014945983887
1.189316749572754
0.9622984528541565
0.7972683906555176
0.6850724816322327
0.6084163188934326
0.5534398555755615
0.5118284821510315
0.478971004486084
0.4521176218986511


In [44]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words:
  
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

In [45]:
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [46]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator = g)
W1 = torch.randn((6, 100), generator = g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [47]:
for p in parameters:
    p.requires_grad = True

In [48]:
for _ in range(20):
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    print(loss.item())
    for p in parameters:
        p.grad = None
    loss.backward()
    for p in parameters:
        p.data += -0.1 * p.grad

19.505229949951172
17.084484100341797
15.776530265808105
14.833341598510742
14.002605438232422
13.253263473510742
12.579920768737793
11.983104705810547
11.47049617767334
11.051857948303223
10.709590911865234
10.407633781433105
10.12781047821045
9.864367485046387
9.61450481414795
9.37644100189209
9.148945808410645
8.931111335754395
8.7222318649292
8.521750450134277
